<a href="https://colab.research.google.com/github/kavithagtamil-create/Model_Building_CV_GK/blob/main/plant_disease_Expert_Model_pil_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plant Disease Detector

Upload photos of leaves. The computer learns to tell healthy leaves from diseased ones,
and gives farming advice.

**How to use this notebook:** click the play button on each cell, top to bottom. Wait for the
green tick before moving to the next one.

There are 5 steps. It takes about 5 minutes.

## Step 1 — Install one tool

This gives us the web page for the final demo. Run it and wait for the tick.

In [ ]:
!pip install -q gradio

## Step 2 — Upload your photos

First, on your own computer, arrange your photos like this and **zip the whole folder**:

```
my_photos/
    healthy/          <- put healthy leaf photos here
    early_blight/     <- put early blight photos here
    late_blight/      <- put late blight photos here
```

The folder names are important — they become the names the computer learns.
Aim for at least 30 photos in each folder.

Now run this cell and choose your zip file.

In [ ]:
from google.colab import files
import zipfile

uploaded = files.upload()
zip_name = list(uploaded)[0]

with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/photos")

print("Done. Photos are uploaded.")

Saving archive.zip to archive.zip
Done. Photos are uploaded.


## Step 3 — Open the photos

This looks inside your zip, finds each folder, and opens the photos.

It prints how many photos it found in each folder. **Check that number before moving on.**

In [ ]:
import pathlib
import numpy as np
from PIL import Image

PHOTOS_PER_FOLDER = 60   # increase this later if you want better results

def photos_in(folder):
    return sorted([f for f in folder.iterdir()
                   if f.suffix.lower() in [".jpg", ".jpeg", ".png"]])

# find every folder that has at least 10 photos in it
folders = [f for f in pathlib.Path("/content/photos").rglob("*")
           if f.is_dir() and len(photos_in(f)) >= 10]

NAMES = [f.name for f in folders]
images = []
labels = []

for number, folder in enumerate(folders):
    chosen = photos_in(folder)[:PHOTOS_PER_FOLDER]
    for photo in chosen:
        picture = Image.open(photo).convert("RGB").resize((224, 224))
        images.append(np.array(picture))
        labels.append(number)
    print(f"{folder.name}: {len(chosen)} photos")

images = np.array(images)
labels = np.array(labels)

print()
print(f"Total: {len(images)} photos, {len(NAMES)} types")

Potato___Early_blight: 60 photos
Potato___Late_blight: 60 photos
Potato___healthy: 60 photos
Potato___Early_blight: 60 photos
Potato___Late_blight: 60 photos
Potato___healthy: 60 photos
Potato___Early_blight: 60 photos
Potato___Late_blight: 60 photos
Potato___healthy: 60 photos

Total: 540 photos, 9 types


## Step 4 — Teach the computer

Two things happen here.

First, a ready-made program called MobileNet looks at each photo and writes down what it sees —
colours, edges, patterns. Someone else already trained it on millions of photos, so we don't
have to. We just borrow it.

Second, we teach a small, simple model to connect those descriptions to your folder names.

At the end it prints a score out of 100. That score comes from photos it has never seen
before, so it is an honest score.

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# MobileNet describes each photo
mobilenet = MobileNetV2(include_top=False, pooling="avg", weights="imagenet")
descriptions = mobilenet.predict(preprocess_input(images.astype("float32")), verbose=0)

# keep some photos aside for testing
train_x, test_x, train_y, test_y = train_test_split(
    descriptions, labels, test_size=0.2, stratify=labels, random_state=42)

# teach the small model
brain = LogisticRegression(max_iter=2000)
brain.fit(train_x, train_y)

score = brain.score(test_x, test_y)
print(f"Accuracy on unseen photos: {score:.0%}")

/tmp/ipykernel_988/2541695404.py:6: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  mobilenet = MobileNetV2(include_top=False, pooling="avg", weights="imagenet")


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Accuracy on unseen photos: 48%


## Step 5 — Try it out

This creates a web page. It prints a link ending in **gradio.live** — open that link on your
phone and upload a leaf photo.

The advice below is written by hand. Change the words to match your crop and your folder names.

If the computer is less than 70% sure, it refuses to answer instead of guessing. That is
deliberate — wrong farming advice is worse than no advice.

In [ ]:
import gradio as gr

ADVICE = {
    "healthy":      "Leaf looks healthy. No spray needed. Keep checking every week.",
    "early_blight": "Early blight. Spray Mancozeb, 2 grams per litre of water. Repeat after 10 days.",
    "late_blight":  "Late blight. Urgent - spray Metalaxyl plus Mancozeb, 2.5 grams per litre today. This spreads fast.",
}

def check_leaf(photo):
    picture = Image.fromarray(photo).convert("RGB").resize((224, 224))
    description = mobilenet.predict(preprocess_input(np.array([picture], dtype="float32")), verbose=0)
    chances = brain.predict_proba(description)[0]

    best = NAMES[chances.argmax()]
    sure = chances.max()

    if sure < 0.70:
        advice = "Not sure. Take a clearer photo of one leaf against a plain background."
    else:
        advice = ADVICE.get(best, "No advice written for this type yet.")

    results = {NAMES[i]: float(chances[i]) for i in range(len(NAMES))}
    return results, advice

gr.Interface(
    fn=check_leaf,
    inputs=gr.Image(label="Leaf photo"),
    outputs=[gr.Label(label="What the computer thinks"),
             gr.Textbox(label="Advice for the farmer")],
    title="Plant Disease Detector",
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8e73964a1459524df1.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---

## If something goes wrong

**Step 3 shows 0 folders** — your zip has the photos loose instead of inside named folders.
Re-zip with one folder per disease.

**Accuracy is low (under 70%)** — usually too few photos. Add more, or increase
`PHOTOS_PER_FOLDER` in Step 3 and run Steps 3 to 5 again.

**Accuracy is 100%** — be suspicious. Usually it means the same photo appears in more than one
folder, or all photos of one type came from a single source.

**The gradio.live link stopped working** — it only lives while this notebook is running.
Run Step 5 again to get a new one.

## If you want to go further

- Try your own crop — tomato, chilli, banana. Only Step 2 changes.
- Add a fourth folder for a new disease. Nothing in the code needs editing.
- Change `PHOTOS_PER_FOLDER` from 60 to 200 and see if the score improves.